# PKTD3-TD: Real-Time Checkpoint & Actor Saturation Monitor

This independent diagnostic notebook allows you to monitor training checkpoints in real time as they land on **Google Drive** without interrupting or touching the running `train_colab.ipynb` training kernel.

### How to Use:
1. Open this notebook in a **separate Colab tab** while `train_colab.ipynb` is running.
2. Mount the same Google Drive.
3. Run the diff check cell at any time (e.g. after episode 500, 1000, 1500, etc.).
4. Verify that `mean_abs_diff > 0.01` and the actor is actively learning rather than frozen.

In [ ]:
# Step 1: Mount Google Drive to access ongoing checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Clone repository and ensure package source is on sys.path
import os, sys

repo_dir = "/content/uav_trajectory_rl"
if not os.path.exists(repo_dir):
    !git clone https://github.com/Krishna200608/uav_trajectory_rl.git {repo_dir}
else:
    %cd {repo_dir}
    !git pull

%cd {repo_dir}
!pip install -e . --quiet

# Direct import path fallback
src_path = os.path.join(repo_dir, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("Ready! Module search path includes:", src_path)

In [ ]:
# Step 3: Set the Drive checkpoint path where training is actively writing
import os

CHECKPOINT_DIR = "/content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/run2"

if not os.path.exists(CHECKPOINT_DIR):
    print(f"Directory {CHECKPOINT_DIR} not found. Checking alternate locations...")
    alt_dir = "/content/drive/MyDrive/Uav_trajectory_rl/PKTD3_TD_Checkpoints/run1"
    if os.path.exists(alt_dir):
        print(f"Found run1 at: {alt_dir}")

# List existing checkpoints
!ls -la "{CHECKPOINT_DIR}"

In [ ]:
# Step 4: Evaluate actor outputs on fixed test states across checkpoints
import os, sys

# Ensure package source is unconditionally resolvable
for p in ["/content/uav_trajectory_rl/src", "src", "."]:
    abs_p = os.path.abspath(p)
    if os.path.exists(abs_p) and abs_p not in sys.path:
        sys.path.insert(0, abs_p)

import glob, re, torch
import numpy as np
from uav_trajectory_rl.td3_networks import Actor

# Find all checkpoints currently on disk, sorted by episode number
ckpt_files = glob.glob(f"{CHECKPOINT_DIR}/td3_agent_ep*.pt")
episodes = sorted(int(re.search(r"ep(\d+)", f).group(1)) for f in ckpt_files)
print("Checkpoints found so far:", episodes)

if not episodes:
    print("No episode checkpoints found yet. Re-run this cell once the first checkpoint (e.g. ep500) has saved.")
else:
    # Fixed test states -- same seed every time you run this, so results are comparable
    # across different points in training and across different runs
    rng = np.random.default_rng(999)
    test_states = rng.uniform(-1.0, 1.0, size=(8, 26)).astype(np.float32)  # state_dim=26 for K=10
    states_t = torch.as_tensor(test_states)

    results = {}
    for ep in episodes:
        ckpt = torch.load(f"{CHECKPOINT_DIR}/td3_agent_ep{ep}.pt", map_location="cpu", weights_only=True)
        actor = Actor(state_dim=26, action_dim=3, max_action=1.0)
        actor.load_state_dict(ckpt["actor"])
        actor.eval()
        with torch.no_grad():
            out = actor(states_t).numpy()
        results[ep] = out
        print(f"\n--- ep{ep} (total_updates={ckpt['total_updates']}) ---")
        print(out.round(4))

    if len(episodes) > 1:
        print("\n=== Mean abs change between consecutive checkpoints ===")
        for a, b in zip(episodes, episodes[1:]):
            diff = np.abs(results[b] - results[a]).mean()
            saturated_frac = (np.abs(results[b]) > 0.999).mean()
            print(f"ep{a} -> ep{b}: mean_abs_diff={diff:.6f}, frac_outputs_saturated(|x|>0.999)={saturated_frac:.2f}")

    print("\n=== INTERPRETATION ===")
    if len(episodes) < 2:
        sat_frac = (np.abs(results[episodes[0]]) > 0.999).mean()
        print(f"Found 1 checkpoint (ep{episodes[0]}). Output saturation fraction (|x|>0.999): {sat_frac:.2f}")
        print("Checkpoint-to-checkpoint diff requires at least 2 checkpoints. Re-run this cell once ep1000 arrives on Drive!")
    else:
        last_diff = np.abs(results[episodes[-1]] - results[episodes[-2]]).mean()
        if last_diff < 1e-4:
            print("WARNING: latest two checkpoints are near-identical -- actor may be frozen again. Consider stopping and investigating before letting the full run finish.")
        else:
            print(f"OK: actor output is still changing between checkpoints (mean diff = {last_diff:.4f}) -- training appears active, not frozen.")